---
## Section 1 — Install & Import Dependencies

We use **TensorFlow/Keras** for the deep learning pipeline and **OpenCV** for camera access.
Run the cell below once to install everything.

In [ ]:
# Install and Imports

# import libraries
import os
import zipfile
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

from sklearn.metrics import classification_report, confusion_matrix
import cv2

In [ ]:
# set random seeds
tf.random.set_seed(42)
np.random.seed(42)

# mount google drive for dataset
from google.colab import drive
drive.mount('/content/drive')

# set zip file path
ZIP_PATH = "/content/drive/MyDrive/lego_dataset.zip"

# root extraction folder
ROOT_EXTRACT = "/content/lego_dataset"

# actual dataset folder inside the zip
EXTRACT_PATH = "/content/lego_dataset/lego_dataset"

# extract only if dataset folder doesn't already exist
if os.path.exists(EXTRACT_PATH):
    print("Dataset already extracted")
else:
    os.makedirs(ROOT_EXTRACT, exist_ok=True)

    if os.path.exists(ZIP_PATH):
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(ROOT_EXTRACT)
        print("Dataset extracted")
    else:
        print(f"Zip file not found at {ZIP_PATH}")

# set dataset and model directory
DATASET_DIR = EXTRACT_PATH
MODEL_DIR = "/content/drive/MyDrive/lego_models"
os.makedirs(MODEL_DIR, exist_ok=True)

# define labels and model path variable
LABELS_PATH = os.path.join(MODEL_DIR, "class_labels.json")
MODEL_PATH = os.path.join(MODEL_DIR, "best_lego_model.keras")

# verify the dataset has class subfolders
dataset_path = Path(DATASET_DIR)
if dataset_path.exists():
    class_folders = [
        d for d in dataset_path.iterdir()
        if d.is_dir() and d.name != "__MACOSX" and d.name != "models"
    ]
    print(f"Found {len(class_folders)} class folders")
    for folder in class_folders[:5]:
        images = list(folder.glob("*.*"))
        print(f"   {folder.name}: {len(images)} images")
else:
    print(f"No dataset found at {DATASET_DIR}")

---
## Section 2 — Configuration

All tunable parameters live in one place so you never have to hunt through the notebook.

### Expected dataset structure
```
dataset/
├── 2357_brick_corner_1x2x2_000L.png
├── 3001_brick_2x4_001R.jpg
├── 3003_brick_2x2_003F.png
└── ...
```
The **filename** is the label source. By default, everything **before the first `_` (part number) + the next underscore-delimited word** is combined to form the class name, e.g.:
- `2357_brick_corner_1x2x2_000L.png` → class `2357_brick_corner_1x2x2`  
  *(strips trailing view/index suffix like `_000L`, `_001R`, etc.)*

You can customise `label_from_filename()` in Section 2 if your naming scheme differs.


In [ ]:
# Configuration Settings

# image settings
IMG_SIZE      = (224, 224)
IMG_SHAPE     = IMG_SIZE + (3,)

# training hyperparameters
BATCH_SIZE       = 64
EPOCHS_FROZEN    = 15
EPOCHS_FINE      = 20
LEARNING_RATE    = 1e-3
FINE_TUNE_LR     = 1e-5
VALIDATION_SPLIT = 0.2

os.makedirs(MODEL_DIR,  exist_ok=True)

---
## Section 3 — Dataset Exploration

Before training we should understand what we're working with:
- How many classes (brick types) are there?
- How many images per class?
- Are the classes balanced?

**Class imbalance** (e.g. 500 images of one brick but only 20 of another) can cause the model to become biased toward the majority class. We'll flag this if it's an issue.

In [ ]:
# Dataset Exploration

# discover classes and count images
dataset_path = Path(DATASET_DIR)
assert dataset_path.exists(), f"Dataset folder not found: {DATASET_DIR}"

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class_counts = {}
for class_dir in sorted(dataset_path.iterdir()):
    if class_dir.is_dir() and class_dir.name != 'models':
        images = [f for f in class_dir.iterdir() if f.suffix.lower() in VALID_EXTENSIONS]
        if images:
            class_counts[class_dir.name] = len(images)

CLASS_NAMES  = list(class_counts.keys())
NUM_CLASSES  = len(CLASS_NAMES)
TOTAL_IMAGES = sum(class_counts.values())

print(f"Classes found : {NUM_CLASSES}")
print(f"Total images  : {TOTAL_IMAGES}")
print(f"\nPer-class breakdown:")
max_count = max(class_counts.values()) if class_counts else 1
for cls, count in class_counts.items():
    bar = '█' * max(1, count * 30 // max_count)
    print(f"  {cls:<40} {count:>5}  {bar}")


In [ ]:
# visualise class distribution
fig, ax = plt.subplots(figsize=(12, max(4, NUM_CLASSES * 0.4)))
colors  = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))

bars = ax.barh(CLASS_NAMES, list(class_counts.values()), color=colors)
ax.set_xlabel("Number of images")
ax.set_title("Dataset — Images per LEGO Class")
ax.bar_label(bars, padding=4)

mean_count = np.mean(list(class_counts.values()))
ax.axvline(mean_count, color='red', linestyle='--', label=f'Mean ({mean_count:.0f})')
ax.legend()
plt.tight_layout()
plt.show()

low = {c: v for c, v in class_counts.items() if v < mean_count * 0.5}
if low:
    print('Under-represented classes (consider collecting more images):')
    for c, v in low.items():
        print(f'   {c}: {v} images')
else:
    print('Class distribution looks reasonably balanced.')


In [ ]:
# preview sample images from each class
cols   = min(NUM_CLASSES, 6)
rows   = (NUM_CLASSES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
axes = np.array(axes).flatten()

for i, cls in enumerate(CLASS_NAMES):
    cls_path = dataset_path / cls
    sample   = next(f for f in cls_path.iterdir() if f.suffix.lower() in VALID_EXTENSIONS)
    img      = mpimg.imread(str(sample))
    axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=8)
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Sample Images — One per Class', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


---
## 🔄 Section 4 — Data Preprocessing & Augmentation

### Why preprocess?
- **Resize**: Neural networks need a fixed input size. We resize everything to 224×224.
- **Normalise**: Pixel values (0–255) are scaled to [-1, 1] because that's what MobileNetV2 was pre-trained on. Models train faster and more stably with normalised inputs.

### Why augment?
Data augmentation artificially **multiplies your training data** by applying random, realistic transformations to images at training time. This forces the model to learn *features* (shapes, studs, edges) rather than memorising specific photos.

Augmentations we use:
| Transform | Why it helps |
|---|---|
| Horizontal flip | Bricks look the same left-right reversed |
| Rotation ±20° | Camera angle variation |
| Zoom ±15% | Bricks at different distances |
| Brightness shift | Different lighting conditions |
| Width/height shift | Brick not perfectly centred |

> Augmentation is applied **only to training data**, never to validation — we want validation to reflect real-world data honestly.

In [ ]:
# Data Preprocessing & Augmentation

# Get all subdirectories that contain images (these are your classes)
class_folders = []
for d in dataset_path.iterdir():
    if d.is_dir() and d.name != 'models':
        # Check if directory contains any images
        images = [f for f in d.iterdir() if f.suffix.lower() in VALID_EXTENSIONS]
        if images:
            class_folders.append(d.name)

CLASS_NAMES = sorted(class_folders)
NUM_CLASSES = len(CLASS_NAMES)
print(f"Found {NUM_CLASSES} classes: {CLASS_NAMES[:5]}...")

# Save class mapping early
idx_to_class = {i: name for i, name in enumerate(CLASS_NAMES)}
with open(LABELS_PATH, "w") as f:
    json.dump(idx_to_class, f, indent=2)
print(f"Class indices saved to {LABELS_PATH}")

# Augmentation pipeline
train_datagen = ImageDataGenerator(
    preprocessing_function = tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split       = VALIDATION_SPLIT,
    rotation_range         = 20,
    width_shift_range      = 0.15,
    height_shift_range     = 0.15,
    zoom_range             = 0.15,
    horizontal_flip        = True,
    brightness_range       = [0.7, 1.3],
    fill_mode              = "nearest"
)

val_datagen = ImageDataGenerator(
    preprocessing_function = tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split       = VALIDATION_SPLIT
)

# Create data generators with explicit class list
train_gen = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size  = IMG_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = "categorical",
    classes      = CLASS_NAMES,
    subset       = "training",
    shuffle      = True,
    seed         = 42
)

val_gen = val_datagen.flow_from_directory(
    DATASET_DIR,
    target_size  = IMG_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = "categorical",
    classes      = CLASS_NAMES,
    subset       = "validation",
    shuffle      = False,
    seed         = 42
)

# Verify the class indices match
print(f"\nTrain generator classes: {len(train_gen.class_indices)}")
print(f"Validation generator classes: {len(val_gen.class_indices)}")
print(f"Expected classes: {NUM_CLASSES}")

# Verify the first few class mappings
print("\nClass mappings (first 5):")
for i, (name, idx) in enumerate(list(train_gen.class_indices.items())[:5]):
    print(f"  {idx}: {name}")

print(f"\nTraining batches   : {len(train_gen)}  ({train_gen.samples} images)")
print(f"Validation batches : {len(val_gen)}  ({val_gen.samples} images)")

In [ ]:
# ── Visualise augmentation effect ────────────────────────────────────────────
# Pull one real batch and show 8 augmented versions of the first image.

sample_imgs, sample_labels = next(train_gen)

# De-normalise from [-1,1] back to [0,1] for display
denorm = lambda x: (x + 1.0) / 2.0

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(np.clip(denorm(sample_imgs[i]), 0, 1))
    label_idx = np.argmax(sample_labels[i])
    ax.set_title(idx_to_class[label_idx], fontsize=8)
    ax.axis("off")

plt.suptitle("Augmented Training Samples", fontsize=13)
plt.tight_layout()
plt.show()

---
## 🧠 Section 5 — Model Architecture (Transfer Learning)

### What is transfer learning?
Training a CNN from scratch requires **millions of images** and days of GPU time. Instead we use **MobileNetV2**, a model pre-trained on 1.2 million ImageNet images. Its early layers already know how to detect edges, textures, and shapes — exactly what we need for LEGO bricks.

We add a **custom classification head** on top to predict our specific brick types.

### Two-phase training strategy
```
Phase 1 — Feature extraction  (base frozen)
  Only our new head trains. The base CNN weights don't change.
  → Fast convergence, avoids destroying pre-trained weights early on.

Phase 2 — Fine-tuning  (top layers of base unfrozen)
  We gradually unfreeze the top layers of MobileNetV2 and retrain
  with a very small learning rate. This adapts low-level features
  to our LEGO-specific visual patterns.
```

### Why MobileNetV2?
- Very small and fast — designed for mobile/edge devices (camera on a Pi!)
- Competitive accuracy vs much larger models
- Included in TensorFlow — no extra downloads

In [ ]:
# Model Architecture

# build mobilenetv2 model
def build_model(num_classes: int, img_shape: tuple) -> keras.Model:
    """
    Builds a transfer-learning model:
      MobileNetV2 base (frozen)  +  custom classification head
    """
    # load MobileNetV2 pre-trained on ImageNet.
    base_model = MobileNetV2(
        input_shape = img_shape,
        include_top = False,
        weights     = "imagenet"
    )
    base_model.trainable = False

    # build classification head
    inputs = keras.Input(shape=img_shape, name="image_input")
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(256, name="fc1")(x)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.Activation("relu", name="relu1")(x)
    x = layers.Dropout(0.4, name="drop1")(x)
    x = layers.Dense(128, name="fc2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.Activation("relu", name="relu2")(x)
    x = layers.Dropout(0.3, name="drop2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = keras.Model(inputs, outputs, name="lego_classifier")
    return model, base_model


model, base_model = build_model(NUM_CLASSES, IMG_SHAPE)

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss      = "categorical_crossentropy",
    metrics   = ["accuracy"]
)

model.summary()

print(f"\nModel built successfully with {NUM_CLASSES} output classes")

---
## 🏋️ Section 6 — Phase 1 Training (Feature Extraction)

We train **only the classification head** while the MobileNetV2 base is frozen.

### Training callbacks
| Callback | Purpose |
|---|---|
| `EarlyStopping` | Stops training when validation accuracy stops improving. Prevents overfitting and wasted compute. |
| `ReduceLROnPlateau` | Halves the learning rate when progress stalls. Helps the model converge. |
| `ModelCheckpoint` | Saves the best model to disk automatically. You get the best version, not the last. |

In [30]:
# Phase 1 Traning

callbacks = [
    EarlyStopping(
        monitor   = "val_accuracy",
        patience  = 5,           # wait 5 epochs for improvement before stopping
        restore_best_weights = True,
        verbose   = 1
    ),
    ReduceLROnPlateau(
        monitor   = "val_loss",
        factor    = 0.5,         # multiply LR by 0.5 when plateaued
        patience  = 3,
        min_lr    = 1e-6,
        verbose   = 1
    ),
    ModelCheckpoint(
        filepath  = MODEL_PATH,
        monitor   = "val_accuracy",
        save_best_only = True,
        verbose   = 1
    )
]

# checkpoint for saved models or disconnected runs
if os.path.exists(MODEL_PATH):
    print(f"Phase 1 checkpoint found — loading and skipping training.")
    model = keras.models.load_model(MODEL_PATH)
    base_model = model.layers[1]
    history_phase1 = None
else:
    print("Phase 1: Training classification head (base frozen)...")
    history_phase1 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=EPOCHS_FROZEN, callbacks=callbacks, verbose=1
    )

Phase 1 checkpoint found — loading and skipping training.


---
## 🔧 Section 7 — Phase 2 Fine-Tuning

Now we **unfreeze the top layers** of MobileNetV2 and retrain with a much smaller learning rate.

- Unfreezing too many layers risks destroying pre-trained weights — we only unfreeze the **top 30 layers**.
- We use **10× smaller learning rate** than Phase 1 to make gentle updates.

In [31]:
# Phase 2 Fine-Tuning

FINE_TUNE_PATH = os.path.join(MODEL_DIR, "best_lego_model_finetuned.keras")

# checkpoint for saved models or disconnected runs
if os.path.exists(FINE_TUNE_PATH):
    print(f"Fine-tuned checkpoint found — loading and skipping Phase 2.")
    model = keras.models.load_model(FINE_TUNE_PATH)
    history_phase2 = None
else:
    base_model.trainable = True
    fine_tune_at = len(base_model.layers) - 30
    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False
    print(f"Fine-tuning {sum(1 for l in base_model.layers if l.trainable)}"
          f" of {len(base_model.layers)} base layers")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
        loss="categorical_crossentropy", metrics=["accuracy"]
    )

    fine_callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=5,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=3, min_lr=1e-6, verbose=1),
        ModelCheckpoint(filepath=FINE_TUNE_PATH, monitor="val_accuracy",
                        save_best_only=True, verbose=1),
    ]

    print("Phase 2: Fine-tuning top base layers...")
    history_phase2 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=EPOCHS_FINE, callbacks=fine_callbacks, verbose=1
    )

MODEL_PATH = FINE_TUNE_PATH if os.path.exists(FINE_TUNE_PATH) else MODEL_PATH
print(f"Active model: {MODEL_PATH}")

Fine-tuned checkpoint found — loading and skipping Phase 2.
Active model: /content/drive/MyDrive/lego_models/best_lego_model_finetuned.keras


---
## Section 8 — Training History Plots

Plot accuracy and loss curves for both training phases.  

**What to look for:**
- Training and validation curves should both trend **upward/downward** together.
- A large **gap** between training and validation accuracy signals **overfitting** — the model memorised training data. Fix: more augmentation, more dropout, less data per class.
- Flat validation line despite rising training = definite overfitting.
- Both lines flat/low = **underfitting** — model too simple or LR too low.

In [32]:
# Training History Plots

# merge both phases history together
def merge_histories(h1, h2):
    """Concatenate two Keras history dicts for plotting."""
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history.get(key, [])
    return combined

# guard against missing history (model loaded from checkpoint after disconnect)
if history_phase1 is None and history_phase2 is None:
    print("No training history available, model was loaded from checkpoint.")
    print("Re-run full models from scratch to see plots.")
else:
    hist = merge_histories(history_phase1, history_phase2)
    phase1_len = len(history_phase1.history["accuracy"])
    total_len  = len(hist["accuracy"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, metric, title in zip(
        axes,
        [("accuracy", "val_accuracy"), ("loss", "val_loss")],
        ["Accuracy", "Loss"]
    ):
        train_key, val_key = metric
        ax.plot(hist[train_key],   label=f"Train {title}",      color="steelblue")
        ax.plot(hist[val_key],     label=f"Validation {title}", color="coral",  linestyle="--")
        ax.axvline(phase1_len - 1, color="green", linestyle=":", label="Fine-tune start")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        ax.set_title(f"Training {title}")
        ax.legend()
        ax.grid(alpha=0.3)

    plt.suptitle("Training History — Phase 1 and Phase 2", fontsize=13)
    plt.tight_layout()
    plt.show()

No training history available — model was loaded from checkpoint.
Skipping training curves. Re-run Sections 6 & 7 from scratch to see plots.


---
## Section 9 — Evaluation & Confusion Matrix

We evaluate the **best saved model** on the validation set.

### Metrics explained
| Metric | Meaning |
|---|---|
| **Precision** | Of all bricks the model *predicted* as class X, what fraction actually are X? |
| **Recall** | Of all bricks that *are* class X, what fraction did the model correctly find? |
| **F1-score** | Harmonic mean of precision and recall — the balanced metric |
| **Confusion matrix** | A grid showing which classes get confused with each other |

In [ ]:
# Evaluation of Model

# load best saved model and run inference on the full validation set
best_model = keras.models.load_model(MODEL_PATH)

val_gen.reset()  # rewind generator to start
y_pred_probs = best_model.predict(val_gen, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)   # predicted class index
y_true       = val_gen.classes                    # ground truth class indices

print("\nClassification Report")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion Matrix

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # normalize by row

fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES), max(6, NUM_CLASSES)))
sns.heatmap(
    cm_norm,
    annot    = True,
    fmt      = ".2f",
    cmap     = "Blues",
    xticklabels = CLASS_NAMES,
    yticklabels = CLASS_NAMES,
    ax       = ax
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True",      fontsize=12)
ax.set_title("Confusion Matrix (row-normalised)", fontsize=13)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [33]:
# Misclassified Examples

val_gen.reset()
all_images, all_labels = [], []
for imgs, lbls in val_gen:
    all_images.append(imgs)
    all_labels.append(np.argmax(lbls, axis=1))
    if len(all_images) * BATCH_SIZE >= val_gen.samples:
        break

all_images = np.concatenate(all_images)[:val_gen.samples]
all_labels = np.concatenate(all_labels)[:val_gen.samples]

wrong_idx = np.where(y_pred != all_labels)[0]
denorm    = lambda x: np.clip((x + 1.0) / 2.0, 0, 1)

print(f"Misclassified: {len(wrong_idx)} / {val_gen.samples}")

show_n = min(8, len(wrong_idx))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in zip(wrong_idx[:show_n], axes.flatten()):
    ax.imshow(denorm(all_images[i]))
    ax.set_title(
        f"True: {idx_to_class[all_labels[i]]}\n"
        f"Pred: {idx_to_class[y_pred[i]]}",
        fontsize=8, color="red"
    )
    ax.axis("off")

plt.suptitle("Misclassified Validation Images", fontsize=13)
plt.tight_layout()
plt.show()

NameError: name 'y_pred' is not defined

---
## Section 10 — Export the Model

We export in two formats:
1. **Keras `.keras`** — for reloading in Python / this notebook
2. **TensorFlow SavedModel** — for deployment on servers or TensorFlow Serving

The label mapping (`class_labels.json`) must travel with the model so predictions can be decoded to brick names.

In [ ]:
# Exporting Model

# savedModel format
saved_model_path = os.path.join(MODEL_DIR, "lego_saved_model")
best_model.export(saved_model_path)

print(f"Keras model: {MODEL_PATH}")
print(f"SavedModel: {saved_model_path}")
print(f"Class labels: {LABELS_PATH}")

# verify the labels file
with open(LABELS_PATH) as f:
    labels = json.load(f)
print(f"\nLabel mapping ({len(labels)} classes):")
print(json.dumps(labels, indent=2))

---
## Single Image Inference

Test the model on any image file before connecting the camera.

In [ ]:
# Single Image Inference from Model

def preprocess_image(image_path: str) -> np.ndarray:
    """
    Load an image from disk and prepare it for the model:
    1. Read as RGB (OpenCV default is BGR, so we convert)
    2. Resize to 224×224
    3. Expand dims: (224,224,3) → (1,224,224,3)  — model expects a batch
    4. Apply MobileNetV2 normalisation
    """
    img   = cv2.imread(image_path)
    img   = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img   = cv2.resize(img, IMG_SIZE)
    img   = np.expand_dims(img, axis=0).astype("float32")
    img   = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img


def predict_brick(model, image_path: str, labels: dict, top_k: int = 3):
    """
    Predict the LEGO brick type in an image.
    Returns the top-k predictions with confidence scores.
    """
    img   = preprocess_image(image_path)
    probs = model.predict(img, verbose=0)[0]       # shape: (num_classes,)

    top_idx   = np.argsort(probs)[::-1][:top_k]   # top-k classes
    results   = [(labels[str(i)], float(probs[i])) for i in top_idx]
    return results


#  Test on a sample image
TEST_IMAGE = "IMG_0701.jpg"   # give the path to your own image to test

if os.path.exists(TEST_IMAGE):
    preds = predict_brick(best_model, TEST_IMAGE, labels)

    img_display = cv2.imread(TEST_IMAGE)
    img_display = cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(img_display)
    ax1.set_title("Input Image")
    ax1.axis("off")

    names  = [p[0] for p in preds]
    scores = [p[1] for p in preds]
    colors_bar = ["#e74c3c" if i == 0 else "#3498db" for i in range(len(preds))]
    ax2.barh(names[::-1], scores[::-1], color=colors_bar[::-1])
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("Confidence")
    ax2.set_title(f"Top-{len(preds)} Predictions")
    for i, (n, s) in enumerate(zip(names[::-1], scores[::-1])):
        ax2.text(s + 0.01, i, f"{s:.1%}", va="center")

    plt.tight_layout()
    plt.show()
    print(f"\nBest prediction: {preds[0][0]}  ({preds[0][1]:.1%} confidence)")
else:
    print(f"Test image not found at '{TEST_IMAGE}'.")